In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np

SEED = 42


# Task 01: Data Ingestion, Schema Checks, and Missingness Handling
This notebook is the notebook version of `run_task01_ingestion.py`, using relative paths only and keeping `price` untransformed.

In [ ]:
INPUT_PATH = Path("../../../data/raw/AB_NYC_2019.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
df.shape

In [ ]:
def build_schema_summary(df: pd.DataFrame) -> str:
    column_meanings = {
        "id": "Listing identifier (unique)",
        "name": "Listing title text",
        "host_id": "Host identifier",
        "host_name": "Host display name",
        "neighbourhood_group": "NYC borough",
        "neighbourhood": "Local neighborhood",
        "latitude": "Listing latitude",
        "longitude": "Listing longitude",
        "room_type": "Type of room/listing",
        "price": "Nightly listing price (target)",
        "minimum_nights": "Minimum nights required",
        "number_of_reviews": "Total review count",
        "last_review": "Date of most recent review",
        "reviews_per_month": "Average reviews per month",
        "calculated_host_listings_count": "Host listing count",
        "availability_365": "Days available in a year",
    }

    lines = []
    lines.append("NYC Airbnb 2019 - Schema Log")
    lines.append(f"Rows: {len(df)}")
    lines.append(f"Columns: {len(df.columns)}")
    lines.append("")
    lines.append("column,dtype,non_null_count,unique_count,description")

    for col in df.columns:
        lines.append(
            f"{col},{df[col].dtype},{df[col].notna().sum()},{df[col].nunique(dropna=True)},{column_meanings.get(col, 'N/A')}"
        )

    return "\n".join(lines) + "\n"

schema_log = build_schema_summary(df)
(OUTPUT_DIR / "schema_log.txt").write_text(schema_log, encoding="utf-8")


## Cleaning Decisions (Justified)
- Drop `host_id`: identifier feature with poor generalization value.
- Drop `last_review`: sparse date feature with high missingness.
- Fill `name` and `host_name` with `Unknown` to preserve records.
- Fill `reviews_per_month` missing values with `0` because this represents no reviews yet.
- Remove rows with `price <= 0` as invalid target values.
- Clip `minimum_nights` at the 99th percentile to limit extreme leverage.
- Keep high `price` values unchanged in Task 01; target transformation is deferred to Task 03.

In [ ]:
def iqr_outlier_stats(series: pd.Series) -> tuple[float, float, int]:
    s = series.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((series < lower) | (series > upper)).sum())
    return float(lower), float(upper), count

missing_pct = (df.isna().mean() * 100).round(4)
missing_count = df.isna().sum()

strategies = {
    "id": ("keep", "No missing values."),
    "name": ("fill with 'Unknown'", "Text field with very low missingness; preserve row count."),
    "host_id": ("drop", "Identifier only; near-unique and not useful as a predictive feature."),
    "host_name": ("fill with 'Unknown'", "Low missingness in host name text; preserve rows."),
    "neighbourhood_group": ("keep", "No missing values."),
    "neighbourhood": ("keep", "No missing values."),
    "latitude": ("keep", "No missing values."),
    "longitude": ("keep", "No missing values."),
    "room_type": ("keep", "No missing values."),
    "price": ("keep", "Target variable; no missing values. Do not transform at ingestion."),
    "minimum_nights": ("clip at 99th percentile", "Extreme values likely represent rare/non-standard listings and can dominate models."),
    "number_of_reviews": ("keep", "No missing values."),
    "last_review": ("drop", "High missingness (~20.56%), date-like field and sparse for never-reviewed listings."),
    "reviews_per_month": ("fill missing with 0", "Missingness means no reviews yet; zero is semantically meaningful."),
    "calculated_host_listings_count": ("keep", "No missing values."),
    "availability_365": ("keep", "No missing values."),
}

report = pd.DataFrame({
    "column": df.columns,
    "missing_count": [int(missing_count[c]) for c in df.columns],
    "missing_pct": [float(missing_pct[c]) for c in df.columns],
    "handling_strategy": [strategies[c][0] for c in df.columns],
    "justification": [strategies[c][1] for c in df.columns],
})
report.to_csv(OUTPUT_DIR / "missingness_report.csv", index=False)

outlier_lines = ["Outlier and Suspicious Value Flags", ""]
numeric_cols = [
    "price", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count", "availability_365"
]

for col in numeric_cols:
    lower, upper, n_outliers = iqr_outlier_stats(df[col])
    p99 = float(df[col].dropna().quantile(0.99))
    outlier_lines.append(f"- {col}: IQR bounds=({lower:.3f}, {upper:.3f}), outlier_count={n_outliers}, p99={p99:.3f}")

non_positive_price = int((df["price"] <= 0).sum())
non_positive_min_nights = int((df["minimum_nights"] <= 0).sum())
outlier_lines.append("")
outlier_lines.append(f"- Suspicious rule check: price <= 0 count = {non_positive_price}")
outlier_lines.append(f"- Suspicious rule check: minimum_nights <= 0 count = {non_positive_min_nights}")
outlier_lines.append("")
outlier_lines.append("Cleaning decisions")
outlier_lines.append("- Drop host_id (identifier) and last_review (sparse date feature).")
outlier_lines.append("- Remove rows where price <= 0 as invalid target values.")
outlier_lines.append("- Fill name/host_name missing values with 'Unknown'.")
outlier_lines.append("- Fill reviews_per_month missing values with 0 (no reviews).")
outlier_lines.append("- Clip minimum_nights above 99th percentile to reduce leverage of extreme values.")
outlier_lines.append("- Keep high price outliers for now because price is the target and should not be transformed in Task 01.")

(OUTPUT_DIR / "outlier_flags.txt").write_text("\n".join(outlier_lines) + "\n", encoding="utf-8")


In [ ]:
clean_df = df.copy()
clean_df = clean_df.drop(columns=["host_id", "last_review"])
clean_df["name"] = clean_df["name"].fillna("Unknown")
clean_df["host_name"] = clean_df["host_name"].fillna("Unknown")
clean_df["reviews_per_month"] = clean_df["reviews_per_month"].fillna(0.0)
clean_df = clean_df[clean_df["price"] > 0].copy()
min_nights_cap = clean_df["minimum_nights"].quantile(0.99)
clean_df["minimum_nights"] = np.minimum(clean_df["minimum_nights"], min_nights_cap)
clean_df.to_csv(OUTPUT_DIR / "ingested.csv", index=False)

assert clean_df.isna().sum().sum() == 0
assert (clean_df["price"] <= 0).sum() == 0
clean_df.shape

In [ ]:
print("Saved:")
print(OUTPUT_DIR / "schema_log.txt")
print(OUTPUT_DIR / "missingness_report.csv")
print(OUTPUT_DIR / "outlier_flags.txt")
print(OUTPUT_DIR / "ingested.csv")
